# 04 · 流式输出（Streaming）

**对应章节**：LangGraph 教程第 04 章 —— 多种流式模式：节点更新流、自定义流、流转换器。

**演示概念**：
- `stream_events(version="v3")` 的自定义事件流（`promptChainStream.py`）；
- `stream(stream_mode=["updates","custom"])` 同时拿到节点更新和自定义流（`promptChainCustomStream.py`）；
- `StreamTransformer` 自定义投影器，把 `custom` 事件重写成 `reasoning` / `content` 类型（`promptChainCustomTrans.py`）；
- `stream_json`（来自 `src.agent_cookbook`，源自 demos 的 `app/llm/streamHelper.py`）把“思考 + 回答”流式写回。

**运行前置**：
- 需要 `.env`（含有效 API Key），放在仓库根目录（与 .env.example 同位置）；
- 需要已 `uv sync`；
- 三个示例都会真实调用模型并流式输出。

## 公共头部：引入 LLM 客户端与流式工具

本 notebook 需要 `structured`、`openaiClient`、`config` 以及 `stream_json`（均来自 `src.agent_cookbook`）。
其中 `openai_client` 以 `openaiClient` 别名引入，以兼容下方 demo 源码里对 `openaiClient` 的引用。

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # 仓库根目录，使 src 包可被导入
from src.agent_cookbook import structured, openai_client as openaiClient, config, stream_json

## 获取所有stream mode的event

In [ ]:
def extract_node(chunk):
    """从任意类型的 chunk 中提取节点名称"""
    t = chunk["type"]
    data = chunk.get("data", {})

    if t == "updates":
        # data = {"node_name": {...}}
        return list(data.keys())[0] if data else None

    elif t == "debug":
        # debug 事件通常包含 "node" 字段
        return data.get("node")

    elif t == "tasks":
        # tasks 事件包含 "name" 字段（任务对应的节点名）
        return data.get("name")

    elif t == "values":
        # values 事件没有节点信息，返回 None
        return None

    elif t == "checkpoints":
        # checkpoints 事件也没有直接节点名，可尝试从 metadata 中提取？
        # 这里简单返回 None
        return data.get("config").get("")

    else:
        return None

In [22]:
from typing import TypedDict
from langgraph.graph import START,StateGraph,END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.runnables import RunnableConfig
from langgraph.types import interrupt

class Message(TypedDict):
    msg : str

def agent_node(state: Message):
    return {"msg": "msg node"}

def tools_node(state: Message):
    #respond = interrupt({"message":state.get("msg")})
    return {"msg": "tool node"}


graph = StateGraph(Message).add_node("agent_node",agent_node).add_node("tool_node",tools_node).add_edge(START,"agent_node").add_edge("agent_node","tool_node").add_edge("tool_node",END)
workflow = graph.compile(MemorySaver())

config = RunnableConfig(configurable={"thread_id":"1"})
for chunks in workflow.stream({"msg":"init"},config=config,stream_mode=["checkpoints","debug","tasks","updates","values"],version="v2"):
    node = extract_node(chunks)
    print(f"node: {node!s:10}, eventtype: {chunks['type']:10}, datas: {chunks['data']}")
    #print(f'node: {list(chunks["data"].keys())[0]}, eventtype: {chunks["type"]}, datas: {chunks["data"]}')

node: None      , eventtype: checkpoints, datas: {'config': {'configurable': {'checkpoint_ns': '', 'thread_id': '1', 'checkpoint_id': '1f1870cb-164f-6126-bfff-274cff48e647'}}, 'parent_config': None, 'values': {}, 'metadata': {'source': 'input', 'step': -1, 'parents': {}}, 'next': ['__start__'], 'tasks': [{'id': '637f1daf-ac4c-3c70-2cec-c3bc71bc3e28', 'name': '__start__', 'interrupts': (), 'state': None}]}
node: None      , eventtype: debug     , datas: {'step': -1, 'timestamp': '2026-07-24T03:06:41.076064+00:00', 'type': 'checkpoint', 'payload': {'config': {'configurable': {'checkpoint_ns': '', 'thread_id': '1', 'checkpoint_id': '1f1870cb-164f-6126-bfff-274cff48e647'}}, 'parent_config': None, 'values': {}, 'metadata': {'source': 'input', 'step': -1, 'parents': {}}, 'next': ['__start__'], 'tasks': [{'id': '637f1daf-ac4c-3c70-2cec-c3bc71bc3e28', 'name': '__start__', 'interrupts': (), 'state': None}]}}
node: None      , eventtype: values    , datas: {'msg': 'init'}
node: None      , event

In [ ]:
import operator
from typing import TypedDict
from langgraph.graph import START,StateGraph,END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.runnables import RunnableConfig
from typing import Annotated

class Message(TypedDict):
    msg : Annotated[list[str],operator.add]
    result: str

def agent_node(state: Message):
    return {"msg": ["msg node"]}

def tools_node(state: Message):
    return {"msg": ["tool node"],"result":"finished"}

newgraph = StateGraph(Message).add_node("agent_node",agent_node).add_node("tool_node",tools_node).add_edge(START,"agent_node").add_edge("agent_node","tool_node").add_edge("tool_node",END)
workflow = newgraph.compile(MemorySaver())

config = RunnableConfig(configurable={"thread_id":"1"})
for chunks in workflow.stream({"msg": ["init"],"result":""},config=config,stream_mode=["tasks","updates","values"],version="v2"):
    #print(f"eventtype: {chunks['type']}, datas: {chunks['data']}")
    if chunks["type"] == "values" or chunks["type"] == "updates":
        print(f"eventtype: {chunks['type']}, datas: {chunks['data']}")
    if chunks["type"] == "tasks":
        print(f"eventtype: {chunks['type']}, datas: {chunks['data'].get("input") if chunks['data'].get("input") else chunks['data'].get("result")}")


print(f'==============')

stream_v3 = workflow.stream_events({"msg": ["init"],"result":""},version="v3")
for snapshot in stream_v3.values:
    print(snapshot)



eventtype: values, datas: {'msg': ['init'], 'result': ''}
eventtype: values, datas: {'msg': ['init'], 'result': ''}
eventtype: tasks, datas: {'id': '6018ed36-a30f-5412-56bd-235d367d766b', 'name': 'agent_node', 'input': {'msg': ['init'], 'result': ''}, 'triggers': ('branch:to:agent_node',)}
eventtype: tasks, datas: {'msg': ['init'], 'result': ''}
eventtype: updates, datas: {'agent_node': {'msg': ['msg node']}}
eventtype: updates, datas: {'agent_node': {'msg': ['msg node']}}
eventtype: tasks, datas: {'id': '6018ed36-a30f-5412-56bd-235d367d766b', 'name': 'agent_node', 'error': None, 'result': {'msg': ['msg node']}, 'interrupts': []}
eventtype: tasks, datas: {'msg': ['msg node']}
eventtype: values, datas: {'msg': ['init', 'msg node'], 'result': ''}
eventtype: values, datas: {'msg': ['init', 'msg node'], 'result': ''}
eventtype: tasks, datas: {'id': 'd002e13d-22ce-afde-4542-80f71cda9a78', 'name': 'tool_node', 'input': {'msg': ['init', 'msg node'], 'result': ''}, 'triggers': ('branch:to:tool

## 1) 自定义事件流：`promptChainStream.py`

依赖 `structured` / `openaiClient` / `config`。`genJoke` 节点直接走 OpenAI 流式接口，
用 `get_stream_writer()` 把 reasoning / content 作为 `custom` 事件写出；外层用
`workflow.stream_events(..., version="v3")` 消费这些事件。

In [ ]:
from typing import TypedDict, Optional, Literal
from langgraph.graph import START, StateGraph, END
from langgraph.config import get_config
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import sys
import json
from langgraph.config import get_stream_writer
load_dotenv()


class Joke(BaseModel):
    joke: str


class CriticResult(BaseModel):
    isFunny: bool = Field(description="is joke funny or not")
    opinion: str = Field(description="how to improve")


class AgentState(TypedDict):
    topic: str
    content: str
    reviewResult: Optional[CriticResult]
    retryCount: int


def genJoke(state: AgentState):
    message: str = f'gen a Joke about {state["topic"]},respond in json format'
    if state["reviewResult"]:
        message = message + f', consider the opinion: {state["reviewResult"].opinion}'

    print(f'# gen \n gen message: {message}')

    stream = openaiClient.chat.completions.create(
        model=config["OPENAI_MODEL"],
        messages=[{"role": "user", "content": message}],
        stream=True,
        response_format={"type": "json_object"},
    )

    answer_content = ""
    in_thinking = False

    for chunk in stream:
        delta = chunk.choices[0].delta if chunk.choices else None
        if not delta:
            continue

        # 处理 thinking/reasoning 内容
        reasoning = getattr(delta, "reasoning_content", None) or ""
        if reasoning:
            if not in_thinking:
                in_thinking = True
                sys.stdout.write("\n🤔 ")
            sys.stdout.write(reasoning)
            sys.stdout.flush()

        # 处理 answer 内容
        content = delta.content or ""
        if content:
            if in_thinking:
                in_thinking = False
                sys.stdout.write("\n\n💬 ")
            sys.stdout.write(content)
            sys.stdout.flush()
            answer_content += content

    print()

    # response_format 保证 answer_content 是合法 JSON，直接解析
    try:
        joke = Joke.model_validate(json.loads(answer_content))
    except (json.JSONDecodeError, ValueError):
        joke = Joke(joke=answer_content.strip())

    return {"content": joke.joke}


def reviewJoke(state: AgentState):
    response = structured(CriticResult, [
            {"role":"system","content":"You are a strict joke reviewer."},
            {"role":"user","content":f'is this Joke funny or not ? Joke: {state["content"]},\
             if not let me know how to improve it'}
        ])
    print(f'## review: \n is funny: {response.isFunny}, opinion: {response.opinion}')
    return {"reviewResult": response, "retryCount": state["retryCount"] + 1}


def checkReviewResult(state: AgentState) -> Literal["genJoke", "translate"]:
    config = get_config()
    max_retries = config.get("configurable", {}).get("max_retries", 3)

    if state["reviewResult"] and state["reviewResult"].isFunny:
        print("✅ 评审通过")
        return "translate"

    if state["retryCount"] >= max_retries:
        print(f"⚠️ 达到最大重试次数 {max_retries}，强制继续")
        return "translate"

    print(f"❌ 评审未通过，重试第 {state['retryCount'] + 1}/{max_retries} 次")
    return "genJoke"


def translate(state: AgentState):
    response = structured(Joke, [{"role":"user","content":f"translate to chinese: {state['content']}"}])
    return {"content": response.joke}


graph = StateGraph(AgentState).add_node("genJoke", genJoke)\
                              .add_node("review", reviewJoke)\
                              .add_node("translate", translate)\
                              .add_edge(START, "genJoke")\
                              .add_edge("genJoke", "review")\
                              .add_conditional_edges(
                                  "review", checkReviewResult,
                                  {"genJoke": "genJoke", "translate": "translate"})\
                              .add_edge("translate", END)

workflow = graph.compile()

stream = workflow.stream_events({"topic":"wednesday","content":"","reviewResult":None,"retryCount":0},
                         {"configurable": {"thread_id": "foo", "max_retries": 2}},version="v3")

for snapshot in stream.values:
        print(f'###:snapshot: {snapshot}')

## 2) updates + custom 双模式：`promptChainCustomStream.py`

依赖 `structured` / `openaiClient` / `config` / `stream_json`。用
`workflow.stream(..., stream_mode=["updates","custom"])`：
- `custom` 取 LLM 的 reasoning / content 交互；
- `updates` 取每个节点的状态更新。

In [ ]:
from typing import TypedDict, Optional, Literal
from langgraph.graph import START, StateGraph, END
from langgraph.config import get_config
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import sys
import json
load_dotenv()


class Joke(BaseModel):
    joke: str


class CriticResult(BaseModel):
    isFunny: bool = Field(description="is joke funny or not")
    opinion: str = Field(description="how to improve")


class AgentState(TypedDict):
    topic: str
    content: str
    reviewResult: Optional[CriticResult]
    retryCount: int


def genJoke(state: AgentState):
    message: str = f'gen a Joke about {state["topic"]},respond in json format'
    if state["reviewResult"]:
        message = message + f', consider the opinion: {state["reviewResult"].opinion}'

    #print(f'# gen \n gen message: {message}')

    joke = stream_json(openaiClient,message,Joke,config["OPENAI_MODEL"]) # type: ignore
    return {"content": joke.joke}


def reviewJoke(state: AgentState):
    response = structured(CriticResult, [
            {"role":"system","content":"You are a strict joke reviewer."},
            {"role":"user","content":f'is this Joke funny or not ? Joke: {state["content"]},\
             if not let me know how to improve it'}
        ])
    #print(f'## review: \n is funny: {response.isFunny}, opinion: {response.opinion}')
    return {"reviewResult": response, "retryCount": state["retryCount"] + 1}


def checkReviewResult(state: AgentState) -> Literal["genJoke", "translate"]:
    config = get_config()
    max_retries = config.get("configurable", {}).get("max_retries", 3)

    if state["reviewResult"] and state["reviewResult"].isFunny:
        print("✅ 评审通过")
        return "translate"

    if state["retryCount"] >= max_retries:
        print(f"⚠️ 达到最大重试次数 {max_retries}，强制继续")
        return "translate"

    print(f"❌ 评审未通过，重试第 {state['retryCount'] + 1}/{max_retries} 次")
    return "genJoke"


def translate(state: AgentState):
    response = structured(Joke, [{"role":"user","content":f"translate to chinese: {state['content']}"}])
    return {"content": response.joke}


graph = StateGraph(AgentState).add_node("genJoke", genJoke)\
                              .add_node("review", reviewJoke)\
                              .add_node("translate", translate)\
                              .add_edge(START, "genJoke")\
                              .add_edge("genJoke", "review")\
                              .add_conditional_edges(
                                  "review", checkReviewResult,
                                  {"genJoke": "genJoke", "translate": "translate"})\
                              .add_edge("translate", END)

workflow = graph.compile()

for chunk in workflow.stream({"topic":"wednesday","content":"","reviewResult":None,"retryCount":0},
                         {"configurable": {"thread_id": "foo", "max_retries": 2}},stream_mode=["updates","custom"],version="v2"):
    if chunk["type"] == "custom":
        # 从custom中取LLM的交互
        reasoning = chunk["data"].get("reasoning","")
        if reasoning:
            sys.stdout.write(reasoning)
            sys.stdout.flush()

    if chunk["type"] == "updates":
        # 从updates取每个node的更新
        print(f'updates: {chunk["data"]}')

## 3) 自定义流转换器：`promptChainCustomTrans.py`

依赖 `structured` / `openaiClient` / `config` / `stream_json`。自定义 `OpenAICustomProjector(StreamTransformer)`，
把 `custom` 事件重投影为 `{"type":"reasoning"/"content", "content": ...}`，再用
`workflow.stream(..., version="v3")` 消费。

In [ ]:
from typing import TypedDict, Optional, Literal
from langgraph.graph import START, StateGraph, END
from langgraph.config import get_config
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import sys
load_dotenv()


from langgraph.stream import StreamTransformer, StreamChannel

class OpenAICustomProjector(StreamTransformer):
    def init(self):
        return StreamChannel()          # 表示这是一个可迭代的输出流

    def process(self, event):
        # event 结构：{'type': 'custom', 'data': {'reasoning': '...'}}
        data = event.get('data')
        if not isinstance(data, dict):
            return None

        if 'reasoning' in data:
            return {'type': 'reasoning', 'content': data['reasoning']}
        elif 'content' in data:
            return {'type': 'content', 'content': data['content']}
        return None                     # 忽略其他事件

    def finalize(self):
        pass

    def fail(self, exc):
        raise exc                       # 或 logging

class Joke(BaseModel):
    joke: str


class CriticResult(BaseModel):
    isFunny: bool = Field(description="is joke funny or not")
    opinion: str = Field(description="how to improve")


class AgentState(TypedDict):
    topic: str
    content: str
    reviewResult: Optional[CriticResult]
    retryCount: int


def genJoke(state: AgentState):
    message: str = f'gen a Joke about {state["topic"]},respond in json format'
    if state["reviewResult"]:
        message = message + f', consider the opinion: {state["reviewResult"].opinion}'

    #print(f'# gen \n gen message: {message}')

    joke = stream_json(openaiClient,message,Joke,config["OPENAI_MODEL"]) # type: ignore
    return {"content": joke.joke}


def reviewJoke(state: AgentState):
    response = structured(CriticResult, [
            {"role":"system","content":"You are a strict joke reviewer."},
            {"role":"user","content":f'is this Joke funny or not ? Joke: {state["content"]},\
             if not let me know how to improve it'}
        ])
    #print(f'## review: \n is funny: {response.isFunny}, opinion: {response.opinion}')
    return {"reviewResult": response, "retryCount": state["retryCount"] + 1}


def checkReviewResult(state: AgentState) -> Literal["genJoke", "translate"]:
    config = get_config()
    max_retries = config.get("configurable", {}).get("max_retries", 3)

    if state["reviewResult"] and state["reviewResult"].isFunny:
        print("✅ 评审通过")
        return "translate"

    if state["retryCount"] >= max_retries:
        print(f"⚠️ 达到最大重试次数 {max_retries}，强制继续")
        return "translate"

    print(f"❌ 评审未通过，重试第 {state['retryCount'] + 1}/{max_retries} 次")
    return "genJoke"


def translate(state: AgentState):
    response = structured(Joke, [{"role":"user","content":f"translate to chinese: {state['content']}"}])
    return {"content": response.joke}


graph = StateGraph(AgentState).add_node("genJoke", genJoke)\
                              .add_node("review", reviewJoke)\
                              .add_node("translate", translate)\
                              .add_edge(START, "genJoke")\
                              .add_edge("genJoke", "review")\
                              .add_conditional_edges(
                                  "review", checkReviewResult,
                                  {"genJoke": "genJoke", "translate": "translate"})\
                              .add_edge("translate", END)

workflow = graph.compile(transformers=[OpenAICustomProjector()])

for chunk in workflow.stream({"topic":"wednesday","content":"","reviewResult":None,"retryCount":0},
                         {"configurable": {"thread_id": "foo", "max_retries": 2}},version="v3"):
    if chunk["type"] == "custom":
        # 从custom中取LLM的交互
        reasoning = chunk["data"].get("reasoning","")
        if reasoning:
            sys.stdout.write(reasoning)
            sys.stdout.flush()

    if chunk["type"] == "updates":
        # 从updates取每个node的更新
        print(f'updates: {chunk["data"]}')

### 小结
- `stream_events` / `stream` 的 `version` 与 `stream_mode` 决定你能拿到什么粒度的数据；
- `get_stream_writer()` 是节点内向外层推送自定义流的标准方式；
- `StreamTransformer` 让你统一重写流的“形状”，方便前端展示思考过程和最终答案。